In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import random

# Configuración del dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

In [ ]:
def add_gaussian_noise(imgs, sigma=0.1):
    noise = torch.randn_like(imgs) * sigma
    return torch.clamp(imgs + noise, 0., 1.)

def add_salt_and_pepper_noise(imgs, amount=0.05):
    noisy_imgs = imgs.clone()
    batch_size, channels, h, w = noisy_imgs.size()
    for i in range(batch_size):
        prob = torch.rand(h, w)
        salt = prob < (amount / 2)
        pepper = (prob > (amount / 2)) & (prob < amount)
        for c in range(channels):
            noisy_imgs[i, c, salt] = 1.0
            noisy_imgs[i, c, pepper] = 0.0
    return noisy_imgs

def convert_to_grayscale(imgs):
    return transforms.Grayscale(num_output_channels=3)(imgs)

def apply_inpainting_mask(imgs, mask_size=12):
    masked_imgs = imgs.clone()
    h, w = imgs.shape[2], imgs.shape[3]
    x = random.randint(0, w - mask_size)
    y = random.randint(0, h - mask_size)
    masked_imgs[:, :, y:y+mask_size, x:x+mask_size] = 0.0
    return masked_imgs

In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self, latent_dim=64):
        super(ConvAutoencoder, self).__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 16, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )
        self.fc_encode = nn.Linear(64 * 4 * 4, latent_dim)
        self.fc_decode = nn.Linear(latent_dim, 64 * 4 * 4)
        # Decoder
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (64, 4, 4)),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(16, 3, 3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        latent = self.fc_encode(x)
        x = self.fc_decode(latent)
        return self.decoder(x)

In [ ]:
def train_experiment(model, dataloader, task, param_value, epochs=10):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()
    train_losses = []

    for epoch in range(epochs):
        running_loss = 0.0
        for imgs, _ in dataloader:
            targets = imgs.to(device)
            
            if task == 'gaussian': inputs = add_gaussian_noise(imgs, sigma=param_value)
            elif task == 'sp': inputs = add_salt_and_pepper_noise(imgs, amount=param_value)
            elif task == 'color': inputs = convert_to_grayscale(imgs)
            elif task == 'inpaint': inputs = apply_inpainting_mask(imgs, mask_size=param_value)
            
            inputs = inputs.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        epoch_loss = running_loss / len(dataloader)
        train_losses.append(epoch_loss)
    return train_losses

def visualize_experiment(model, dataloader, task, param_value, title, num_imgs=3):
    model.eval()
    imgs, _ = next(iter(dataloader))
    
    if task == 'gaussian': inputs = add_gaussian_noise(imgs, sigma=param_value)
    elif task == 'sp': inputs = add_salt_and_pepper_noise(imgs, amount=param_value)
    elif task == 'color': inputs = convert_to_grayscale(imgs)
    elif task == 'inpaint': inputs = apply_inpainting_mask(imgs, mask_size=param_value)
        
    with torch.no_grad():
        outputs = model(inputs.to(device)).cpu()
    
    fig, axes = plt.subplots(3, num_imgs, figsize=(9, 6))
    fig.suptitle(title, fontsize=16)
    for i in range(num_imgs):
        axes[0, i].imshow(np.transpose(imgs[i].numpy(), (1, 2, 0)))
        axes[0, i].axis('off')
        if i == 0: axes[0, i].set_title("Original")
        
        axes[1, i].imshow(np.transpose(inputs[i].numpy(), (1, 2, 0)))
        axes[1, i].axis('off')
        if i == 0: axes[1, i].set_title("Entrada Corrupta")
        
        axes[2, i].imshow(np.transpose(outputs[i].numpy(), (1, 2, 0)))
        axes[2, i].axis('off')
        if i == 0: axes[2, i].set_title("Reconstrucción")
    plt.tight_layout()
    plt.show()

In [ ]:
# Diccionario para guardar todas las curvas de pérdida y analizarlas después
history = {}
EPOCHS = 10 # Puedes bajarlo a 5 si quieres hacer una prueba rápida primero

print("=== INICIANDO BATERÍA DE EXPERIMENTOS ===")

# ---------------------------------------------------------
# TAREA 1: Denoising (Comparando niveles y tipos de ruido)
# ---------------------------------------------------------
print("\n--- Tarea 1: Ruido Gaussiano (3 niveles) ---")
sigmas = [0.1, 0.3, 0.5]
history['gaussian'] = {}
for s in sigmas:
    print(f"Entrenando Gaussiano con sigma={s}...")
    model = ConvAutoencoder(latent_dim=64).to(device)
    history['gaussian'][s] = train_experiment(model, trainloader, 'gaussian', s, EPOCHS)
    visualize_experiment(model, testloader, 'gaussian', s, f'Gaussiano Sigma: {s}')

print("\n--- Tarea 1: Ruido Sal y Pimienta (3 niveles) ---")
amounts = [0.05, 0.1, 0.2]
history['sp'] = {}
for a in amounts:
    print(f"Entrenando Sal y Pimienta con amount={a}...")
    model = ConvAutoencoder(latent_dim=64).to(device)
    history['sp'][a] = train_experiment(model, trainloader, 'sp', a, EPOCHS)
    visualize_experiment(model, testloader, 'sp', a, f'Sal y Pimienta Amount: {a}')


# ---------------------------------------------------------
# TAREA 2: Colorization (Comparando Espacios Latentes)
# ---------------------------------------------------------
print("\n--- Tarea 2: Colorización (3 tamaños de Espacio Latente) ---")
latents = [16, 32, 64]
history['color'] = {}
for l in latents:
    print(f"Entrenando Colorización con latent_dim={l}...")
    model = ConvAutoencoder(latent_dim=l).to(device)
    history['color'][l] = train_experiment(model, trainloader, 'color', None, EPOCHS)
    visualize_experiment(model, testloader, 'color', None, f'Colorización Latent Dim: {l}')


# ---------------------------------------------------------
# TAREA 3: Inpainting (Comparando tamaño de región faltante)
# ---------------------------------------------------------
print("\n--- Tarea 3: Inpainting (3 tamaños de máscara) ---")
masks = [8, 12, 16]
history['inpaint'] = {}
for m in masks:
    print(f"Entrenando Inpainting con mask_size={m}...")
    model = ConvAutoencoder(latent_dim=64).to(device)
    history['inpaint'][m] = train_experiment(model, trainloader, 'inpaint', m, EPOCHS)
    visualize_experiment(model, testloader, 'inpaint', m, f'Inpainting Mask Size: {m}x{m}')


# =========================================================
# GRÁFICAS DE ANÁLISIS PARA TUS RESPUESTAS
# =========================================================
print("\n=== GENERANDO GRÁFICAS DE ANÁLISIS ===")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Gráfica 1: ¿Qué tarea fue más difícil? (Comparamos las versiones "medias" de cada tarea)
axes[0].plot(history['gaussian'][0.3], label='Denoising (Gauss 0.3)', marker='o')
axes[0].plot(history['color'][64], label='Colorization (Latent 64)', marker='s')
axes[0].plot(history['inpaint'][12], label='Inpainting (Mask 12)', marker='^')
axes[0].set_title('Comparativa General de Tareas')
axes[0].set_xlabel('Épocas')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()

# Gráfica 2: ¿Qué ruido es más difícil? (Comparamos Gaussiano vs S&P moderados)
axes[1].plot(history['gaussian'][0.3], label='Gaussiano (sigma=0.3)', marker='o', color='red')
axes[1].plot(history['sp'][0.1], label='Sal y Pimienta (amount=0.1)', marker='x', color='purple')
axes[1].set_title('Dificultad por Tipo de Ruido')
axes[1].set_xlabel('Épocas')
axes[1].legend()

# Gráfica 3: Impacto del Espacio Latente (En colorización)
axes[2].plot(history['color'][16], label='Latent = 16', marker='v', linestyle='--')
axes[2].plot(history['color'][32], label='Latent = 32', marker='d', linestyle='-.')
axes[2].plot(history['color'][64], label='Latent = 64', marker='o')
axes[2].set_title('Impacto del Tamaño Latente')
axes[2].set_xlabel('Épocas')
axes[2].legend()

plt.tight_layout()
plt.show()